# Benchmark: Scalability vs Grid Size

Measures how correction runtime, L2 error, and iteration count scale with
increasing grid resolution.  A fixed random DVF pattern is generated at a
small base size and upscaled to each target resolution, keeping the folding
pattern consistent across sizes.

Sizes tested: **10x10, 20x20, 40x40, 60x60, 80x80, 100x100, 150x150, 200x200**

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from dvfopt import (
    iterative_with_jacobians2,
    iterative_parallel,
    jacobian_det2D,
    generate_random_dvf,
    scale_dvf,
)

## Configuration

In [ ]:
GRID_SIZES = [10, 20, 40, 60, 80, 100, 150, 200]

# Base DVF: small field upscaled to each target size
BASE_SHAPE = (3, 1, 5, 5)
MAX_MAGNITUDE = 5.0
SEED = 42

# Use parallel solver for larger grids
USE_PARALLEL = True

## Generate test fields at each resolution

In [ ]:
base_dvf = generate_random_dvf(BASE_SHAPE, MAX_MAGNITUDE, SEED)

test_fields = {}
for size in GRID_SIZES:
    dvf = scale_dvf(base_dvf, (size, size))
    phi = np.stack([dvf[-2, 0], dvf[-1, 0]])
    jac = jacobian_det2D(phi)
    n_neg = int((jac <= 0).sum())
    test_fields[size] = dvf
    print(f"  {size:>4d}x{size:<4d}  neg-Jdet: {n_neg:>5d}  "
          f"min Jdet: {jac.min():+.4f}  total pixels: {size*size}")

## Run corrections

In [ ]:
solver = iterative_parallel if USE_PARALLEL else iterative_with_jacobians2
solver_name = "parallel" if USE_PARALLEL else "serial"
print(f"Solver: {solver_name}\n")

results = {}

print(f"{'Size':>10s}  {'Time (s)':>10s}  {'Neg init':>10s}  {'Neg final':>10s}  "
      f"{'Min Jdet':>10s}  {'L2 Error':>10s}")
print("-" * 70)

for size in GRID_SIZES:
    dvf = test_fields[size]
    phi_init = np.stack([dvf[-2, 0], dvf[-1, 0]])
    jac_init = jacobian_det2D(phi_init)
    n_neg_init = int((jac_init <= 0).sum())

    t0 = time.perf_counter()
    phi = solver(dvf.copy(), verbose=0)
    elapsed = time.perf_counter() - t0

    jac_final = jacobian_det2D(phi)
    n_neg_final = int((jac_final <= 0).sum())
    min_jdet = float(jac_final.min())
    l2 = float(np.sqrt(np.sum((phi - phi_init) ** 2)))

    results[size] = {
        "time": elapsed,
        "n_neg_init": n_neg_init,
        "n_neg_final": n_neg_final,
        "min_jdet": min_jdet,
        "l2_err": l2,
    }

    print(f"{size:>4d}x{size:<4d}  {elapsed:>10.2f}  {n_neg_init:>10d}  "
          f"{n_neg_final:>10d}  {min_jdet:>+10.6f}  {l2:>10.4f}")

---
## Plots

In [ ]:
sizes = list(results.keys())
times = [results[s]["time"] for s in sizes]
l2s = [results[s]["l2_err"] for s in sizes]
negs = [results[s]["n_neg_init"] for s in sizes]
pixels = [s * s for s in sizes]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Runtime vs grid size
ax = axes[0, 0]
ax.plot(sizes, times, "o-", color="tab:blue")
ax.set_xlabel("Grid size (NxN)")
ax.set_ylabel("Time (s)")
ax.set_title("Runtime vs Grid Size")
ax.grid(True, alpha=0.3)

# Runtime vs total pixels (log-log)
ax = axes[0, 1]
ax.loglog(pixels, times, "o-", color="tab:blue")
# Fit power law: time ~ pixels^alpha
log_p = np.log(pixels)
log_t = np.log(times)
alpha, log_c = np.polyfit(log_p, log_t, 1)
fit_t = np.exp(log_c) * np.array(pixels) ** alpha
ax.loglog(pixels, fit_t, "--", color="tab:red",
          label=f"fit: O(n^{alpha:.2f})")
ax.set_xlabel("Total pixels")
ax.set_ylabel("Time (s)")
ax.set_title("Runtime Scaling (log-log)")
ax.legend()
ax.grid(True, alpha=0.3)

# L2 error vs grid size
ax = axes[1, 0]
ax.plot(sizes, l2s, "s-", color="tab:orange")
ax.set_xlabel("Grid size (NxN)")
ax.set_ylabel("L2 Error")
ax.set_title("L2 Deviation vs Grid Size")
ax.grid(True, alpha=0.3)

# Initial neg-Jdet count vs grid size
ax = axes[1, 1]
ax.plot(sizes, negs, "^-", color="tab:green")
ax.set_xlabel("Grid size (NxN)")
ax.set_ylabel("Neg Jdet pixels (initial)")
ax.set_title("Folding Pixels vs Grid Size")
ax.grid(True, alpha=0.3)

plt.suptitle(f"Scalability — {solver_name} solver", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Normalized runtime: time per initial neg-Jdet pixel ---
time_per_neg = [results[s]["time"] / max(results[s]["n_neg_init"], 1) for s in sizes]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([f"{s}x{s}" for s in sizes], time_per_neg, color="tab:purple")
ax.set_ylabel("Time per neg-Jdet pixel (s)")
ax.set_title("Correction Cost per Folding Pixel")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Summary

Key takeaways:
- **Runtime scaling** — the log-log slope gives the empirical complexity exponent
- **L2 error** — larger grids may allow more diffuse corrections (lower per-pixel impact)
- **Cost per pixel** — how much additional work each folding pixel requires at different resolutions